In [18]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error

#load output
df = pd.read_csv('full_dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

#sort by region and time
df = df.sort_values(['region', 'datetime']).reset_index(drop=True)

print('Shape:', df.shape)
print('Regions:', df['region'].unique())

Shape: (35040, 12)
Regions: ['california' 'illinois' 'texas' 'virginia']


#Lag Features
Lag features capture "what was carbon intensity X hours ago?"
We created these PER region so lags don't bleed across regions.

In [24]:
#lag_1 = 1 hour ago, lag_2 = 2 hours ago, etc.
for lag in ["1", "2", "3", "24"]:
    df[f"carbon_intensity_lag_{lag}"] = (df.groupby("region")["carbon_intensity"].shift(int(lag)))

lag_cols = [col for col in df.columns if "lag" in col]
print(df[["datetime", "region", "carbon_intensity"] + lag_cols].head(30))

              datetime      region  carbon_intensity  carbon_intensity_lag_1  \
0  2023-01-02 00:00:00  california            189.75                     NaN   
1  2023-01-02 01:00:00  california            207.79                  189.75   
2  2023-01-02 02:00:00  california            213.71                  207.79   
3  2023-01-02 03:00:00  california            214.20                  213.71   
4  2023-01-02 04:00:00  california            216.54                  214.20   
5  2023-01-02 05:00:00  california            220.28                  216.54   
6  2023-01-02 06:00:00  california            225.27                  220.28   
7  2023-01-02 07:00:00  california            228.60                  225.27   
8  2023-01-02 08:00:00  california            247.55                  228.60   
9  2023-01-02 09:00:00  california            254.81                  247.55   
10 2023-01-02 10:00:00  california            258.92                  254.81   
11 2023-01-02 11:00:00  california      

#Cyclical Time Encodings
Hour, day of week, and month are cyclical. We encode them as sine/cosine pairs to capture this.

In [20]:
#sin/cos encoding captures circularity better than raw numbers
df['hour_sin'] = np.sin(2 * np.pi * df["hour"] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df["hour"] / 24)

df['day_of_week_sin'] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df['month_sin'] = np.sin(2 * np.pi * df["month"] / 12)
df['month_cos'] = np.cos(2 * np.pi * df["month"] / 12)

print('Cyclical time encodings added.')

Cyclical time encodings added.


#Drop rows with NaNs from lag features
Lagging shifts values forward, leaving NaN in the first few rows of each region. We dropped these rows so the model isn't trained on incomplete data.

In [21]:
df = df.dropna().reset_index(drop=True)
print('Shape after dropping NaN rows:', df.shape)

Shape after dropping NaN rows: (34944, 22)


#Train/Test Split by Region
Train on california (CAISO) + texas (ERCOT)
Test on virginia (PJM) - a region the model has never seen. This tests whether the model generalizes to a completely unseen region.

In [25]:
feature_cols = [
    # weather
    "temp_c", "wind_speed", "cloud_cover", "solar_radiation",
    # lag features
    "carbon_intensity_lag_1", "carbon_intensity_lag_2",
    "carbon_intensity_lag_3", "carbon_intensity_lag_24",
    # cyclical time encodings
    "hour_sin", "hour_cos",
    "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos",
    # energy mix
    "cfe_pct", "re_pct"
]

target_col = 'carbon_intensity'

train_df = df[df['region'].isin(['california','texas'])]
test_df = df[df['region'] == 'virginia']

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print('Train samples:', X_train.shape[0])
print('Test samples:', X_test.shape[0])

Train samples: 17472
Test samples: 8736


#Train Ridge Regression Baseline
Ridge regression is trained on california and texas, then evaluated on virginia.
A high R2 and low MAE means the model generalizes well to an unseen region.

In [23]:
#Ridge regression needs StandardScaler since features on different scales would be unfairly penalized
ridge_pipeline = make_pipeline(StandardScaler(), Ridge())
ridge_pipeline.fit(X_train, y_train)

#evaluate
y_pred = ridge_pipeline.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f'Ridge Regression R2 Score: {r2:.4f}')
print(f'Ridge Regression MAE: {mae:.4f} gCO2eq/kWh')

Ridge Regression R2 Score: 0.9087
Ridge Regression MAE: 7.6006 gCO2eq/kWh
